# Compare Gemini RAG profile executions

Loads every `*_exe*.profile.json` file in `data/pdf_original_cache/`, groups them by paper, and diffs each section across runs. Useful for measuring pipeline **stability** -- how consistent are the extractions when the same PDF is analysed multiple times.

Naming convention: `<paper>_exeN.profile.json` (e.g. `2023_04_article_exe1.profile.json`).

## Preamble - pin working directory to `src/`

In [1]:
import os, sys
from pathlib import Path

HERE = Path.cwd()
candidates = [HERE, HERE / "src", *HERE.parents, *(p / "src" for p in HERE.parents)]
SRC = next((c for c in candidates if (c / "gemini_rag.py").exists()), None)
if SRC is None:
    raise RuntimeError("Could not locate src/ containing gemini_rag.py")
os.chdir(SRC)
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
print(f"working directory: {SRC}")

working directory: /home/andres/Documents/21st_publications/src


## 0. Discover profile files

Scans `data/pdf_original_cache/` and groups by paper id.

In [2]:
import json
import re
import pandas as pd

CACHE = Path("data/pdf_original_cache")
EXE_PATTERN = re.compile(r"^(.+)_exe(\d+)\.profile\.json$")

groups: dict[str, dict[int, Path]] = {}
for p in sorted(CACHE.glob("*_exe*.profile.json")):
    m = EXE_PATTERN.match(p.name)
    if m:
        paper_id, exe_n = m.group(1), int(m.group(2))
        groups.setdefault(paper_id, {})[exe_n] = p

overview = pd.DataFrame([
    {"paper": k, "executions": sorted(v.keys()), "n_runs": len(v)}
    for k, v in groups.items()
])
print(f"{len(groups)} paper(s) with multiple executions found\n")
overview

1 paper(s) with multiple executions found



,paper,executions,n_runs
0,2023_04_article,"[1, 2, 3, 4]",4


## 1. Pick a paper to compare

Change `PAPER` to any id from the table above.

In [3]:
PAPER = "2023_04_article"
assert PAPER in groups, f"no executions found for {PAPER}"

exes: dict[int, dict] = {
    n: json.loads(p.read_text()) for n, p in sorted(groups[PAPER].items())
}
print(f"Paper: {PAPER}")
print(f"Title (exe1): {exes[next(iter(exes))].get('title', '')}")
print(f"Runs loaded:  {list(exes.keys())}")

Paper: 2023_04_article
Title (exe1): Object Detection Meets Knowledge Graphs
Runs loaded:  [1, 2, 3, 4]


## 2. High-level counts per section

Quick way to see which executions diverge. If all rows are identical, the pipeline is at least producing the same *shape* of output.

In [4]:
SECTIONS = {
    "authors":              lambda d: len(d.get("authors", [])),
    "methodology_steps":    lambda d: len(d.get("methodology_steps", [])),
    "datasets":             lambda d: len(d.get("datasets", [])),
    "figures":              lambda d: len(d.get("figures_to_reproduce", [])),
    "tables":               lambda d: len(d.get("tables_to_reproduce", [])),
    "hyperparameters":      lambda d: len(d.get("hyperparameters", [])),
    "repository_links":     lambda d: len(d.get("repository_links", [])),
    "supplementary":        lambda d: len(d.get("supplementary_materials", [])),
    "software_versions":    lambda d: len(d.get("stated_software_versions", {})),
}

counts = pd.DataFrame({
    f"exe{n}": {label: fn(d) for label, fn in SECTIONS.items()}
    for n, d in exes.items()
})
counts["min"] = counts.min(axis=1)
counts["max"] = counts.max(axis=1)
counts["delta"] = counts["max"] - counts["min"]
counts

,exe1,exe2,exe3,exe4,min,max,delta
authors,5,5,5,5,5,5,0
methodology_steps,4,4,5,4,4,5,1
datasets,3,3,4,3,3,4,1
figures,4,4,4,4,4,4,0
tables,3,3,3,3,3,3,0
hyperparameters,17,13,13,17,13,17,4
repository_links,1,3,1,1,1,3,2
supplementary,6,0,6,6,0,6,6
software_versions,0,0,0,0,0,0,0


## 3. Set-based comparison per section

For each section that has natural item identifiers (dataset name, figure id, repo URL, etc.) we show which items appear in which execution. A row of all ✓ means fully stable; mixed rows are the divergences the pipeline introduced.

`jaccard_stability` = |intersection| / |union|. 1.0 means every item appears in every run.

In [5]:
def compare_sets(exes: dict, key: str, label_fn) -> pd.DataFrame:
    sets = {n: {label_fn(item) for item in d.get(key, [])} for n, d in exes.items()}
    all_labels = sorted(set().union(*sets.values()))
    rows = []
    for label in all_labels:
        row = {"item": label}
        for n, s in sets.items():
            row[f"exe{n}"] = "yes" if label in s else ""
        rows.append(row)
    df = pd.DataFrame(rows) if rows else pd.DataFrame(columns=["item"])
    return df, sets


def jaccard(sets_dict: dict) -> float:
    values = list(sets_dict.values())
    if not values:
        return 1.0
    u = set().union(*values)
    if not u:
        return 1.0
    i = set(values[0])
    for s in values[1:]:
        i &= s
    return len(i) / len(u)

### 3.1 Datasets (by name)

In [6]:
df, sets = compare_sets(exes, "datasets", lambda d: d.get("name", "").strip())
print(f"jaccard stability: {jaccard(sets):.2f}")
df

jaccard stability: 0.75


,item,exe1,exe2,exe3,exe4
0,ImageNet,,,yes,
1,MIT ConceptNet,yes,yes,yes,yes
2,MSCOCO15,yes,yes,yes,yes
3,PASCAL07,yes,yes,yes,yes


### 3.2 Repository links (by URL)

In [7]:
df, sets = compare_sets(exes, "repository_links", lambda d: d.strip().rstrip("/"))
print(f"jaccard stability: {jaccard(sets):.2f}")
df

jaccard stability: 0.33


,item,exe1,exe2,exe3,exe4
0,http://conceptnet-api-1.media.mit.edu,,yes,,
1,http://mscoco.org/home,,yes,,
2,https://github.com/rbgirshick/py-faster-renn,yes,yes,yes,yes


### 3.3 Figures (by id)

In [8]:
df, sets = compare_sets(exes, "figures_to_reproduce", lambda d: d.get("id", "").strip())
print(f"jaccard stability: {jaccard(sets):.2f}")
df

jaccard stability: 1.00


,item,exe1,exe2,exe3,exe4
0,Figure 1,yes,yes,yes,yes
1,Figure 2,yes,yes,yes,yes
2,Figure 3,yes,yes,yes,yes
3,Figure 4,yes,yes,yes,yes


### 3.4 Tables (by id)

In [9]:
df, sets = compare_sets(exes, "tables_to_reproduce", lambda d: d.get("id", "").strip())
print(f"jaccard stability: {jaccard(sets):.2f}")
df

jaccard stability: 1.00


,item,exe1,exe2,exe3,exe4
0,Table 1,yes,yes,yes,yes
1,Table 2,yes,yes,yes,yes
2,Table 3,yes,yes,yes,yes


## 4. Hyperparameters - name and value drift

Hyperparameters are compared by name. Mismatched values for the same name signal that Gemini pulled the number from different passages -- worth auditing against the `source_quote`.

In [10]:
by_exe = {
    n: {hp.get("name", "").strip(): hp.get("value", "") for hp in d.get("hyperparameters", [])}
    for n, d in exes.items()
}
all_names = sorted(set().union(*(b.keys() for b in by_exe.values())))
rows = []
for name in all_names:
    row = {"hyperparameter": name}
    values = set()
    for n in by_exe:
        v = by_exe[n].get(name, "")
        row[f"exe{n}"] = v
        if v:
            values.add(str(v).strip())
    row["disagree"] = "disagree" if len(values) > 1 else ("missing_in_some" if not all(name in b for b in by_exe.values()) else "")
    rows.append(row)
hp_df = pd.DataFrame(rows)
print(f"name-level jaccard stability: {jaccard({n: set(b.keys()) for n, b in by_exe.items()}):.2f}")
hp_df

name-level jaccard stability: 0.17


,hyperparameter,exe1,exe2,exe3,exe4,disagree
0,IoU threshold (MSCOCO15),"{0.50, 0.55, 0.95}",,,"{0.50, 0.55, 0.95}",missing_in_some
1,IoU threshold (PASCAL07),0.5,,,0.5,missing_in_some
2,IoU threshold for visualization,0.75,,,0.75,missing_in_some
3,epsilon,"{0.1, 0.25, 0.5, 0.75,0.9}","{0.1, 0.25, 0.5, 0.75, 0.9}","{0.1, 0.25, 0.5, 0.75, 0.9}","{0.1, 0.25, 0.5, 0.75,0.9}",disagree
4,iterations,,10,10,,missing_in_some
5,k (nearest neighbors),,5,5,,missing_in_some
6,learning rate,,1e-4,1e-4,,missing_in_some
7,learning rate (initial),1e-3,,,1e-3,missing_in_some
8,learning rate (subsequent),1e-4,,,1e-4,missing_in_some
9,mini-batch size,2,2,2,2,


## 5. Methodology steps side-by-side

Free-text, so no set math -- just a truncated preview to skim. Mismatched counts or wildly different phrasings mean the pipeline decomposed the methodology differently across runs.

In [11]:
max_steps = max(len(d.get("methodology_steps", [])) for d in exes.values())
rows = []
for i in range(max_steps):
    row = {"step": i + 1}
    for n, d in exes.items():
        steps = d.get("methodology_steps", [])
        row[f"exe{n}"] = (steps[i]["description"][:90] + "...") if i < len(steps) else "—"
    rows.append(row)
methodology_df = pd.DataFrame(rows)
methodology_df

,step,exe1,exe2,exe3,exe4
0,1,An existing object detection algorithm (Faster...,An existing object detection algorithm (Faster...,An existing object detection algorithm (Faster...,An existing object detection algorithm (Faster...
1,2,Semantic consistency (S) is computed for each ...,Semantic consistency (S) is computed for each ...,Semantic consistency for pairs of concepts is ...,Semantic consistency (S) is computed for each ...
2,3,Semantic consistency (S) is computed for each ...,Semantic consistency (S) is computed for each ...,Semantic consistency is quantified using a kno...,Semantic consistency (S) is computed for each ...
3,4,The initial object detection probabilities (P)...,The initial object detection probabilities (P)...,The initial object detection probabilities (ma...,The initial object detection probabilities (P)...
4,5,—,—,The performance of the re-optimized object det...,—


## 6. Overall stability report

Single-number summary per section, plus a pipeline-level mean. Closer to 1.0 = more reproducible.

In [12]:
def stability(key: str, label_fn) -> float:
    _, sets = compare_sets(exes, key, label_fn)
    return jaccard(sets)

scores = {
    "datasets":         stability("datasets", lambda d: d.get("name", "").strip()),
    "repository_links": stability("repository_links", lambda d: d.strip().rstrip("/")),
    "figures":          stability("figures_to_reproduce", lambda d: d.get("id", "").strip()),
    "tables":           stability("tables_to_reproduce", lambda d: d.get("id", "").strip()),
    "hyperparam_names": jaccard({n: set(b.keys()) for n, b in by_exe.items()}),
}
scores["overall_mean"] = sum(scores.values()) / len(scores)
pd.Series(scores, name="jaccard_stability").round(3).to_frame()

,jaccard_stability
datasets,0.750
repository_links,0.333
figures,1.000
tables,1.000
hyperparam_names,0.174
overall_mean,0.651


## 7. Export comparison report to JSON

In [13]:
report = {
    "paper": PAPER,
    "executions": list(exes.keys()),
    "counts": counts.drop(columns=["min", "max", "delta"]).to_dict(),
    "stability_jaccard": scores,
    "hyperparameter_diffs": hp_df.to_dict(orient="records"),
}
out = CACHE / f"{PAPER}_comparison.json"
out.write_text(json.dumps(report, indent=2, default=str))
print(f"wrote: {out}")

wrote: data/pdf_original_cache/2023_04_article_comparison.json
